In [1]:
import pandas as pd
import os

# 1) 원본 로드
df = pd.read_excel('전국.xlsx', dtype=str)

# 2) addr1 또는 title이 NaN인 행 삭제
df = df.dropna(subset=['addr1', 'title'])

# 3) '숙박' 카테고리 삭제
df = df[df['cat1'] != '숙박'].copy()

# 4) addr1을 시·도(첫 토큰)로 덮어쓰기
df['addr1'] = df['addr1'].str.split().str[0]

# 5) 시·도 통합 매핑 테이블
sido_map = {
    '서울':        '서울특별시', '서울특별시':  '서울특별시', '서울시':      '서울특별시',
    '부산':        '부산광역시', '부산광역시':  '부산광역시',
    '대구':        '대구광역시', '대구광역시':  '대구광역시',
    '인천':        '인천광역시', '인천광역시':  '인천광역시',
    '광주':        '광주광역시', '광주광역시':  '광주광역시',
    '대전':        '대전광역시', '대전광역시':  '대전광역시',
    '울산':        '울산광역시', '울산광역시':  '울산광역시', '울산시':      '울산광역시',
    '세종':        '세종특별자치시', '세종특별자치시':'세종특별자치시',
    '경기':        '경기도',     '경기도':      '경기도',
    '강원도':      '강원도',     '강원특별자치도':'강원도',
    '충남':        '충청남도',   '충청남도':    '충청남도',
    '충북':        '충청북도',   '충청북도':    '충청북도',
    '전남':        '전라남도',   '전라남도':    '전라남도',
    '전북':        '전라북도',   '전라북도':    '전라북도',
    '전북특별자치도':'전라북도',
    '경남':        '경상남도',   '경상남도':    '경상남도',
    '경북':        '경상북도',   '경상북도':    '경상북도',
    '제주도':      '제주특별자치도', '제주특별자치도':'제주특별자치도',
}

# 6) addr1을 매핑으로 통일, 없으면 '기타'
df['addr1'] = df['addr1'].map(sido_map).fillna('기타')

# 7) 출력 폴더 준비
out_dir = '전국_시도별'
os.makedirs(out_dir, exist_ok=True)

# 8) addr1(통일된 시·도 또는 '기타')별로 분할 후 저장
for sido, grp in df.groupby('addr1'):
    path = os.path.join(out_dir, f"{sido}.xlsx")
    grp.to_excel(path, index=False)
    print(f"↳ {sido}.xlsx ({len(grp)}건) 저장됨")

↳ 강원도.xlsx (4617건) 저장됨
↳ 경기도.xlsx (9070건) 저장됨
↳ 경상남도.xlsx (3574건) 저장됨
↳ 경상북도.xlsx (3220건) 저장됨
↳ 광주광역시.xlsx (546건) 저장됨
↳ 대구광역시.xlsx (1088건) 저장됨
↳ 대전광역시.xlsx (721건) 저장됨
↳ 부산광역시.xlsx (1909건) 저장됨
↳ 서울특별시.xlsx (7334건) 저장됨
↳ 세종특별자치시.xlsx (200건) 저장됨
↳ 울산광역시.xlsx (624건) 저장됨
↳ 인천광역시.xlsx (1897건) 저장됨
↳ 전라남도.xlsx (3010건) 저장됨
↳ 전라북도.xlsx (2282건) 저장됨
↳ 제주특별자치도.xlsx (2080건) 저장됨
↳ 충청남도.xlsx (2306건) 저장됨
↳ 충청북도.xlsx (1803건) 저장됨


In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import pandas as pd
import csv
import sys

# 필드 최대 길이 늘리기 (긴 리뷰 대응)
csv.field_size_limit(2**31 - 1)   # 약 2GB (최대 허용값에 가깝다)

# ──────────────────────────────────────────────
# 설정
# ──────────────────────────────────────────────
DATA_DIR = Path("전국_그룹분리_csv")
REGION = "충청남도"
OUTFILE = DATA_DIR / f"{REGION}_all.csv"

# ──────────────────────────────────────────────
# 대상 파일 수집
# ──────────────────────────────────────────────
file_paths = sorted(DATA_DIR.glob(f"{REGION}*.csv"))
if not file_paths:
    raise FileNotFoundError(f"{REGION}*.csv 파일 없음")

print(f"\n📂 병합 대상 파일 목록 ({len(file_paths)}개):")
for p in file_paths:
    print(f"  • {p.name}")

# ──────────────────────────────────────────────
# 줄 단위 수동 파싱 (깨진 줄 포함)
# ──────────────────────────────────────────────
COLS = ["search_keyword", "place_name", "address", "url", "date", "review_text"]
all_rows = []
bad_lines = []

for p in file_paths:
    with open(p, encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader, None)  # 헤더 스킵
        for i, row in enumerate(reader, start=2):  # line번호=2부터 시작
            if len(row) != 6:
                bad_lines.append((p.name, i, len(row)))
            # 길이 보정: 너무 적으면 빈칸 채우고, 너무 많으면 자름
            fixed = (row + [""] * 6)[:6]
            all_rows.append(fixed)

# ──────────────────────────────────────────────
# DataFrame 생성 및 후처리
# ──────────────────────────────────────────────
df = pd.DataFrame(all_rows, columns=COLS)

before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f"\n🧹 중복 제거: {before - after:,}행 삭제 → {after:,}행 남음")

df.sort_values(by="address", ascending=False).to_csv(
    OUTFILE,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)

print(f"\n✅ 저장 완료 → {OUTFILE}")

# ──────────────────────────────────────────────
# 깨진 줄 보고
# ──────────────────────────────────────────────
if bad_lines:
    print(f"\n⚠️ 깨진 줄 {len(bad_lines)}개 (열 개수 안 맞음):")
    for fname, line_num, colcount in bad_lines[:10]:
        print(f"  - {fname} line {line_num} → {colcount}개 열")
    if len(bad_lines) > 10:
        print(f"  ... 외 {len(bad_lines) - 10}개 생략")
else:
    print("\n🎉 깨진 줄 없이 모두 정상 처리됨!")



📂 병합 대상 파일 목록 (3개):
  • 충청남도.csv
  • 충청남도2.csv
  • 충청남도3.csv

🧹 중복 제거: 183행 삭제 → 109,185행 남음

✅ 저장 완료 → 전국_그룹분리_csv\충청남도_all.csv

🎉 깨진 줄 없이 모두 정상 처리됨!


In [10]:
import pandas as pd

df = pd.read_csv("전국_그룹분리_csv\충청남도_all.csv", dtype=str)

unique_rows = df.drop_duplicates(subset="search_keyword")
print(len(unique_rows))


2302


In [11]:
import pandas as pd
import csv

# 파일 경로 (Windows 백슬래시는 r'' 권장)
path = r"전국_그룹분리_csv\충청남도_all.csv"

# 1) 깨진 줄(열 개수 불일치) 스캔
bad_lines = []
with open(path, "r", encoding="utf-8-sig", newline="") as f:
    r = csv.reader(f)
    header = next(r)  # 헤더 1행
    expected = len(header)
    for row in r:
        if len(row) != expected:
            # csv.reader.line_num = 해당 레코드를 마친 물리 라인 번호
            bad_lines.append((r.line_num, len(row)))

print(f"[헤더 열 수] expected = {expected}")
if bad_lines:
    print(f"⚠️ 깨진 줄 {len(bad_lines)}개 발견(열 개수 불일치):")
    for ln, cols in bad_lines:
        print(f"  - line {ln} → {cols}개 열")
else:
    print("✅ 열 개수 기준으로는 깨진 줄 없음.")

# 2) DataFrame 로딩(깨진 줄은 건너뛰고 읽기)
#    pandas 1.4+ : on_bad_lines 사용
df = pd.read_csv(path, dtype=str, encoding="utf-8-sig", on_bad_lines="skip")
print("정상 로드된 행 수:", len(df))


[헤더 열 수] expected = 6
✅ 열 개수 기준으로는 깨진 줄 없음.
정상 로드된 행 수: 109185


In [10]:
import pandas as pd

# 1) 파일 읽기 (헤더가 이미 있을 경우 header=0)
df = pd.read_csv('법정동코드 전체자료.txt',
                sep='\t', 
                encoding='cp949',
                header=0,
                dtype=str,      # 코드 앞의 0 보존
                engine='python' # C엔진에서 드물게 일어나는 파싱 에러 회피
            )

# 2. 제대로 분리되었는지 확인


# 3) UTF‑8 SIG 로 다시 저장 (Excel에서 깨짐 방지)
df.to_csv(
    '법정동_분리된_데이터.csv',
    sep=',',             # csv 로 저장
    index=False,
    encoding='utf-8-sig'
)
# # 또는
# df.to_excel('법정동_분리된_데이터.xlsx', index=False)
df.head()

,법정동코드,법정동명,폐지여부
0,1100000000,서울특별시,존재
1,1111000000,서울특별시 종로구,존재
2,1111010100,서울특별시 종로구 청운동,존재
3,1111010200,서울특별시 종로구 신교동,존재
4,1111010300,서울특별시 종로구 궁정동,존재


In [ ]:
import re
import logging
import requests
from math import radians, sin, cos, atan2, sqrt
import pandas as pd
from konlpy.tag import Okt
from transformers import pipeline, logging as hf_logging
import json
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

# HuggingFace 로거 레벨 설정 (불필요한 경고 메시지 숨김)
hf_logging.set_verbosity_error()

# -----------------------------
# 설정 (기존과 동일)
# -----------------------------
REGION_SHORT = "울산"
metro_map = {
    "서울": "서울특별시", "부산": "부산광역시", "대구": "대구광역시",
    "인천": "인천광역시", "광주": "광주광역시", "대전": "대전광역시",
    "울산": "울산광역시", "세종": "세종특별자치시", "경기": "경기도",
    "강원": "강원특별자치도", "충북": "충청북도", "충남": "충청남도",
    "전북": "전북특별자치도", "전남": "전라남도", "경북": "경상북도", "경남": "경상남도",
    "제주": "제주특별자치도"
}
REGION_FULL   = metro_map[REGION_SHORT]
ALL_CSV       = f"전국_그룹분리_csv/{REGION_SHORT}_all.csv"
FILTERED_CSV  = f"전국_그룹분리_csv/{REGION_SHORT}_filtered.csv"
DELETED_CSV   = f"전국_그룹분리_csv/{REGION_SHORT}_deleted.csv"
GEO_FILE      = f"전국_시도별/{REGION_FULL}.xlsx"
CACHE_FILE    = "api_geocode_cache.json"
API_KEY       = "0be64775eb0d51574480225b2b175243"  # 🚨 실제 카카오 API 키를 입력하세요.
HEADERS       = {"Authorization": f"KakaoAK {API_KEY}"}
KEYWORD_URL   = "https://dapi.kakao.com/v2/local/search/keyword.json"

# -----------------------------
# 0) 로깅 설정
# -----------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")

# -----------------------------
# 1) 전역 변수 및 헬퍼 함수 준비
# -----------------------------
logging.info("전역 변수 및 헬퍼 함수를 준비합니다.")
# 법정동 데이터 로드
law_df = pd.read_csv("법정동_분리된_데이터.csv", encoding="utf-8-sig")
law_names = set(law_df["법정동명"])

# 법정동 토큰맵 생성
okt_for_law = Okt()
law_token_map = defaultdict(set)
for name in law_names:
    for tk in okt_for_law.nouns(name):
        law_token_map[tk].add(name)

# 지역별 주소-좌표 매핑
try:
    geo_df = pd.read_excel(GEO_FILE)
    geo_df["full_addr"] = geo_df["addr1"].fillna("") + " " + geo_df["addr2"].fillna("")
    addr_to_coord = dict(zip(geo_df["full_addr"], zip(geo_df["mapy"], geo_df["mapx"])))
except FileNotFoundError:
    logging.warning(f"{GEO_FILE}을 찾을 수 없습니다. API 조회에 더 의존합니다.")
    addr_to_coord = {}


# NLP 모델 및 정규식
okt = Okt()
pattern = re.compile(r'.*(시|도|군|구|읍|면|동|역|터미널|공원)$')
regex_pd = re.compile(r'([가-힣]+?)(?:광역시|특별시|도|시)?\s*([가-힣]+(?:군|구|읍|면))')
try:
    ner = pipeline("ner", model="monologg/koelectra-base-v3-discriminator", grouped_entities=True)
except Exception as e:
    logging.warning(f"NER 모델 로딩 실패: {e}. NER 기능 없이 계속합니다.")
    ner = None

def geocode_keyword(place):
    """카카오 키워드 검색 API를 통해 좌표를 가져오는 함수"""
    try:
        r = requests.get(KEYWORD_URL, headers=HEADERS, params={"query": place}, timeout=5)
        r.raise_for_status()
        docs = r.json().get("documents", [])
        if docs:
            return float(docs[0]["y"]), float(docs[0]["x"])
    except requests.exceptions.RequestException as e:
        logging.error(f"API 요청 실패: {place}, 오류: {e}")
    return None, None

# -----------------------------
# 2) 핵심 로직 함수 (이전 답변과 동일)
# -----------------------------
def process_reviews(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """리뷰 데이터프레임을 받아 필터링/삭제된 데이터프레임을 반환하는 메인 함수"""
    
    logging.info("단계 1: 모든 리뷰 텍스트에서 장소명 일괄 추출을 시작합니다.")
    texts = df["review_text"].fillna("").tolist()
    
    ner_results = []
    if ner:
        logging.info(f"{len(texts)}개의 텍스트에 대해 NER 일괄 처리를 수행합니다.")
        non_empty_texts = [t for t in texts if t]
        if non_empty_texts:
            ner_outputs = ner(non_empty_texts, batch_size=16)
            ner_map = {text: [e["word"].replace(" ", "") for e in output if e.get("entity_group") == "LOC"] for text, output in zip(non_empty_texts, ner_outputs)}
            ner_results = [ner_map.get(t, []) for t in texts]
        else:
            ner_results = [[] for _ in texts]
    else:
        ner_results = [[] for _ in texts]

    def extract_and_resolve(text: str, ner_mentions: list) -> str | None:
        if not text:
            return None
        mset = set()
        for prov, dist in regex_pd.findall(text):
            prov_full = metro_map.get(prov, prov if prov.endswith(("시", "도")) else prov + "도")
            mset.add(f"{prov_full} {dist}")
        nouns = okt.nouns(text)
        mset.update(n for n in nouns if pattern.match(n))
        mset.update(ner_mentions)
        exact_matches = [m for m in mset if m in law_names]
        if exact_matches:
            return min(exact_matches, key=len)
        mentions = [m for m in mset if m in law_token_map]
        if mentions:
            mentions.sort(key=lambda t: len(law_token_map[t]))
            candidates = law_token_map[mentions[0]].copy()
            for token in mentions[1:]:
                candidates &= law_token_map[token]
                if not candidates:
                    break
            if candidates:
                return min(candidates, key=len)
        if ner_mentions:
            return max(ner_mentions, key=len)
        return None

    logging.info("장소명 추출 및 법정동명 결정을 시작합니다...")
    mention_places = [extract_and_resolve(text, ner_res) for text, ner_res in zip(texts, ner_results)]
    df["mention_place"] = mention_places
    logging.info("✅ 장소명 추출 및 법정동명 결정이 완료되었습니다.")

    logging.info("단계 2: API 조회를 위한 좌표 캐싱 및 병렬 요청을 수행합니다.")
    try:
        with open(CACHE_FILE, 'r', encoding='utf-8') as f:
            api_cache = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        api_cache = {}

    unique_places = df["mention_place"].dropna().unique()
    to_fetch = [p for p in unique_places if p not in addr_to_coord and p not in api_cache]

    if to_fetch:
        logging.info(f"총 {len(to_fetch)}개의 새로운 장소명에 대한 좌표를 API로 조회합니다.")
        with ThreadPoolExecutor(max_workers=10) as executor:
            results = executor.map(geocode_keyword, to_fetch)
            for place, coords in zip(to_fetch, results):
                if coords[0] is not None:
                    api_cache[place] = coords
        with open(CACHE_FILE, 'w', encoding='utf-8') as f:
            json.dump(api_cache, f, ensure_ascii=False, indent=2)

    logging.info("단계 3: 좌표 매핑 및 최종 필터링을 수행합니다.")
    full_coord_map = {**api_cache, **addr_to_coord}
    coords = df["mention_place"].map(full_coord_map).apply(pd.Series)
    coords.columns = ['lat', 'lon']
    df = pd.concat([df, coords], axis=1)

    is_empty_review = df['review_text'].fillna('').str.strip() == ''
    is_wrong_region = df["mention_place"].notna() & ~df["mention_place"].str.startswith(REGION_FULL, na=False)
    is_unresolved_place = df['mention_place'].isna()
    is_deleted = is_empty_review | is_wrong_region | is_unresolved_place
    
    filtered_df = df[~is_deleted].reset_index(drop=True)
    deleted_df = df[is_deleted].reset_index(drop=True)
    
    return filtered_df, deleted_df

# -----------------------------
# 3) 메인 실행 로직
# -----------------------------
if __name__ == "__main__":
    logging.info("리뷰 데이터 파일을 불러옵니다.")
    try:
        all_df = pd.read_csv(ALL_CSV, encoding="utf-8-sig")
    except FileNotFoundError:
        logging.error(f"오류: 원본 파일 '{ALL_CSV}'을 찾을 수 없습니다. 파일 경로를 확인해주세요.")
        exit()
        
    # ✨✨✨✨✨[수정된 부분]✨✨✨✨✨
    # 원본 CSV의 'place_name' 열을 코드에서 사용할 'store_name'으로 변경합니다.
    if 'place_name' in all_df.columns:
        all_df.rename(columns={'place_name': 'store_name'}, inplace=True)
        logging.info("'place_name' 열을 'store_name'으로 변경했습니다.")
    else:
        # 'place_name'도 없는 경우를 대비하여 빈 'store_name' 열 생성
        if 'store_name' not in all_df.columns:
            logging.warning(f"경고: 'place_name' 또는 'store_name' 열을 원본 파일에서 찾을 수 없습니다.")
            all_df['store_name'] = ""
            
    all_df["orig_row"] = all_df.index + 2

    # 핵심 로직 실행
    filtered_df, deleted_df = process_reviews(all_df.copy())

    # 결과 저장
    logging.info(f"결과를 CSV 파일로 저장합니다.")
    final_cols = ['store_name', 'address', 'review_text', 'mention_place', 'lat', 'lon', 'orig_row']
    
    filtered_df.reindex(columns=final_cols).to_csv(FILTERED_CSV, index=False, encoding="utf-8-sig")
    deleted_df.reindex(columns=final_cols).to_csv(DELETED_CSV, index=False, encoding="utf-8-sig")

    logging.info(f"✅ 필터링 완료 – 남은 {len(filtered_df)}건, 삭제된 {len(deleted_df)}건.")

12:08:42 전역 변수 및 헬퍼 함수를 준비합니다.
c:\Users\hyunj\anaconda3\Lib\site-packages\transformers\pipelines\token_classification.py:181: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(
12:11:05 리뷰 데이터 파일을 불러옵니다.
12:11:16 'place_name' 열을 'store_name'으로 변경했습니다.
12:11:16 단계 1: 모든 리뷰 텍스트에서 장소명 일괄 추출을 시작합니다.
12:11:16 28260개의 텍스트에 대해 NER 일괄 처리를 수행합니다.
15:49:51 장소명 추출 및 법정동명 결정을 시작합니다...
18:05:42 ✅ 장소명 추출 및 법정동명 결정이 완료되었습니다.
18:05:42 단계 2: API 조회를 위한 좌표 캐싱 및 병렬 요청을 수행합니다.
18:05:42 총 29개의 새로운 장소명에 대한 좌표를 API로 조회합니다.
18:05:43 단계 3: 좌표 매핑 및 최종 필터링을 수행합니다.
18:05:58 결과를 CSV 파일로 저장합니다.
18:06:06 ✅ 필터링 완료 – 남은 16745건, 삭제된 11515건.
